# create a new datasets_csv folder with the original ds minus the data missing from wsi


In [1]:
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, random_split

mkdir -p failed for path /.config/matplotlib: [Errno 13] Permission denied: '/.config'
Matplotlib created a temporary cache directory at /tmp/matplotlib-qzd1bch6 because there was an issue with the default path (/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
from src.clean_datasets import clean_datasets_clinical, clean_datasets_label, clean_datasets_omics

In [3]:
# stuff to remove

In [4]:
# import datasets to clean
# omics data
omics_data_c = pd.read_csv(
    os.path.join('src/datasets_csv_original/raw_rna_data/combine/brca', 'rna_clean.csv'),    # all rna data
    engine='python', 
    index_col=0
)

omics_data_h = pd.read_csv(
    os.path.join('src/datasets_csv_original/raw_rna_data/hallmarks/brca', 'rna_clean.csv'),    # all rna data
    engine='python', 
    index_col=0
)

omics_data_x = pd.read_csv(
    os.path.join('src/datasets_csv_original/raw_rna_data/xena/brca', 'rna_clean.csv'),    # all rna data
    engine='python', 
    index_col=0
)

# metadata
label_data = pd.read_csv('src/datasets_csv_original/metadata/tcga_brca.csv', low_memory=False)    # contains pairing between case_id and slide_id(s) + info on survival

# clinical data
clinical_data = pd.read_csv('src/datasets_csv_original/clinical_data/tcga_brca_clinical.csv', index_col=0) # clinical data


In [5]:
label_data = clean_datasets_label(label_data)
omics_data_c = clean_datasets_omics(omics_data_c)
omics_data_h = clean_datasets_omics(omics_data_h)
omics_data_x = clean_datasets_omics(omics_data_x)
clinical_data = clean_datasets_clinical(clinical_data)

##### clean label_data #####
Number of case_ids to delete: 223
Number of rows in label_data before changes: 931
Number of rows to delete from label_data: 272
Number of rows in label_data after changes: 659
##### clean omics_data #####
Number of rows in omics_data before changes: 939
Number of rows to delete from omics_data: 212
Number of rows in omics_data after changes: 727
##### clean omics_data #####
Number of rows in omics_data before changes: 939
Number of rows to delete from omics_data: 212
Number of rows in omics_data after changes: 727
##### clean omics_data #####
Number of rows in omics_data before changes: 939
Number of rows to delete from omics_data: 212
Number of rows in omics_data after changes: 727
##### clean clinical_data #####
Number of rows in clinical_data before changes: 1108
Number of rows to delete from clinical_data: 212
Number of rows in clinical_data after changes: 896


### check before saving

In [13]:
id_label = label_data['case_id'].values
print('ids label size: ', np.unique(id_label).shape)
id_omics_c = omics_data_c.index.values
print('ids omics_c size: ', np.unique(id_omics_c).shape)
id_clinical = clinical_data['case_id'].values
print('ids clinical size: ', np.unique(id_clinical).shape)

ids label size:  (659,)
ids omics_c size:  (723,)
ids clinical size:  (889,)


In [15]:
# get stuff that repeats, explore a bit
# some case_ids repeat themselves in omics_data. get the case_ids that have multiple lines
unique, counts = np.unique(id_omics_c, return_counts=True)
id_omics_c_counts = dict(zip(unique, counts))
repeated_case_ids = [k for k, v in id_omics_c_counts.items() if v > 1]
print('repeated_case_ids omics_c: ', len(repeated_case_ids))
print('repeated case_ids: ', repeated_case_ids)


repeated_case_ids omics_c:  4
repeated case_ids:  ['TCGA-AC-A6IX', 'TCGA-BH-A1FE', 'TCGA-E2-A15A', 'TCGA-E2-A15K']


In [16]:
# check if the repeated case_ids are in label_data
for case_id in repeated_case_ids:
    if case_id in id_label:
        print(case_id)

TCGA-AC-A6IX
TCGA-BH-A1FE
TCGA-E2-A15A
TCGA-E2-A15K


some patients have multiple lines in omics data: the index (case_id) is the same but the values for the other attributes are different. In clinical data we find that the patients are registered twice with different 'subtype' values, ex ILC, BREAST...

There is also more data on patients (more patients) for omics (and clinical) than wsi_slides. 

In [6]:
# save data into new datasets_csv
omics_data_c.to_csv(os.path.join('src/datasets_csv/raw_rna_data/combine/brca', 'rna_clean.csv'))
omics_data_h.to_csv(os.path.join('src/datasets_csv/raw_rna_data/hallmarks/brca', 'rna_clean.csv'))
omics_data_x.to_csv(os.path.join('src/datasets_csv/raw_rna_data/xena/brca', 'rna_clean.csv'))
label_data.to_csv('src/datasets_csv/metadata/tcga_brca.csv')
clinical_data.to_csv('src/datasets_csv/clinical_data/tcga_brca_clinical.csv')
